# Merchant Attrition Early-Warning

Build a **grounded, self-verifying** agent that flags **which merchants are decaying toward churn** by category — and contextualizes that decline against the category's **demand backdrop** (US Census retail sales), so you separate a real attrition problem from a whole-category slowdown.

| Step | What you build |
|---|---|
| 1 | Explore the benchmark (`ATTRITION_BENCHMARK`) |
| 2 | Query the **semantic view** (Cortex Analyst) |
| 3 | Build a verified **fact pack** |
| 4 | **Grounded draft** under DATA RULES |
| 5 | **Verify** claims vs evidence |
| 6 | Persist the save-list note |

**Prereq:** run `setup_synthetic.sql` (no listing) or `setup.sql` (real Census category demand). Set the notebook role to one that owns `SNOWPACK_MERCHANT_ATTRITION.CORE`.

In [ ]:
SCHEMA = "SNOWPACK_MERCHANT_ATTRITION.CORE"
MODEL  = "claude-sonnet-4-5"
import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()
mode = session.sql(f"select mode, note from {SCHEMA}.ATTRITION_DATA_SOURCE_INFO").collect()[0]
print("data mode:", mode[0], "—", mode[1])

## Step 1 — Explore the benchmark

`ATTRITION_BENCHMARK` = merchant health rolled up to **(category, month, risk_band, tenure_band)**, joined to the category demand backdrop. `risk_band` ∈ healthy / watch / at-risk / churned; NRR = volume retained vs baseline.

In [ ]:
session.sql(f"""
  SELECT category_name, risk_band,
         SUM(merchant_count)  AS merchants,
         ROUND(SUM(monthly_volume_usd)/1e6,1) AS volume_m,
         ROUND(AVG(avg_volume_decline_pct),1) AS avg_decline_pct,
         ROUND(AVG(category_yoy_growth_pct),1) AS cat_demand_yoy
  FROM {SCHEMA}.ATTRITION_BENCHMARK
  WHERE month_end = (SELECT MAX(month_end) FROM {SCHEMA}.ATTRITION_BENCHMARK)
  GROUP BY category_name, risk_band
  ORDER BY category_name, risk_band
""").to_pandas()

## Step 2 — Cortex Analyst over the semantic view

`ATTRITION_SV` lets Analyst answer NL questions like *"which categories have the most at-risk merchants in 2026?"*. Direct `SEMANTIC_VIEW()` query below; the agent writes this for you from plain English. `churn_rate_pct` and `net_revenue_retention` are percentages; category metrics are non-additive (AVG).

In [ ]:
session.sql(f"""
  SELECT * FROM SEMANTIC_VIEW({SCHEMA}.ATTRITION_SV
      METRICS at_risk_merchants, churn_rate_pct, net_revenue_retention, category_demand_yoy
      DIMENSIONS category_name, report_year)
  WHERE report_year = 2026 ORDER BY at_risk_merchants DESC
""").to_pandas()

## Step 3 — Verified fact pack

Latest-month health per category — the agent's only allowed numbers: active vs churned merchants, churn rate, NRR, volume decline, and the category demand backdrop.

In [ ]:
fact_pack = session.sql(f"""
  WITH latest AS (SELECT MAX(month_end) z FROM {SCHEMA}.ATTRITION_BENCHMARK)
  SELECT ARRAY_AGG(OBJECT_CONSTRUCT(
           'category', g.category_name, 'month', g.month_end::string,
           'total_merchants', g.total_merchants, 'active_merchants', g.active_merchants,
           'churned_merchants', g.churned_merchants, 'at_risk_merchants', g.at_risk_merchants,
           'churn_rate_pct', g.churn_rate_pct, 'net_revenue_retention_pct', g.nrr_pct,
           'avg_volume_decline_pct', g.avg_vol_decline, 'category_yoy_growth_pct', g.cat_yoy))
  FROM (
    SELECT b.category_name, b.month_end,
           SUM(b.merchant_count)        AS total_merchants,
           SUM(b.active_merchant_count) AS active_merchants,
           SUM(b.churned_count)         AS churned_merchants,
           SUM(b.at_risk_count)         AS at_risk_merchants,
           ROUND(SUM(b.churned_count)*100.0/NULLIF(SUM(b.merchant_count),0),1)        AS churn_rate_pct,
           ROUND(SUM(b.monthly_volume_usd)*100.0/NULLIF(SUM(b.baseline_volume_usd),0),1) AS nrr_pct,
           ROUND(AVG(b.avg_volume_decline_pct),1) AS avg_vol_decline,
           ROUND(AVG(b.category_yoy_growth_pct),1) AS cat_yoy
    FROM {SCHEMA}.ATTRITION_BENCHMARK b JOIN latest ON b.month_end = latest.z
    GROUP BY b.category_name, b.month_end
  ) g
""").collect()[0][0]
print(fact_pack[:700], '...')

## Step 4 — Grounded draft (DATA RULES)

The model may use ONLY the fact pack — no invented numbers, no hedging.

In [ ]:
DATA_RULES = (
    "You are a payments merchant-retention analyst. Write a concise attrition early-warning note for the processor using ONLY the FACTS JSON.\n"
    "DATA RULES: use only numbers in FACTS, copy exactly; churn_rate_pct and net_revenue_retention_pct are percentages; avg_volume_decline_pct positive = volume below baseline; "
    "category_yoy_growth_pct is the category demand backdrop; contrast merchant attrition against it (a merchant falling while its category grows is a real save-list target); dollar fields are USD; no estimates, no hedging words, no outside benchmarks; missing = n/d.\n"
    "OUTPUT markdown starting with '# ', a one-line takeaway (where attrition is worst), a table (category | active | churned | at-risk | churn rate % | NRR % | category YoY %), then 3 bullets prioritizing a save-list, ending with '## Data Coverage' (source + window)."
)
prompt = DATA_RULES + f"\n\nFACTS: {fact_pack}"
draft_md = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, prompt]).collect()[0][0]
print(draft_md)

## Step 5 — Verify against the evidence

A second pass fact-checks every number against the fact pack and flags hedges — the quality gate.

In [ ]:
verify_prompt = (
    "You are a fact-checker. Compare the DRAFT against the FACTS JSON (the only allowed source). "
    "Return ONLY a valid JSON object (double quotes, no code fences, no prose) with keys: status (\"verified\" if every number "
    "and claim is supported by FACTS and there are no hedge words like likely/approximately/around/could, else \"needs_review\"), "
    "residual_hedges (integer), edits (array of short strings naming each unsupported or hedged claim).\n\n"
    f"FACTS: {fact_pack}\n\nDRAFT:\n{draft_md}"
)
raw = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, verify_prompt]).collect()[0][0]
import re as _re
def _parse_verdict(x):
    t = x.strip()
    for _ in range(3):
        m = _re.match(r"^```[a-zA-Z]*\s*(.*?)\s*```$", t, _re.S)
        if m:
            t = m.group(1).strip()
        try:
            v = json.loads(t)
        except Exception:
            a, b = t.find("{"), t.rfind("}")
            if a >= 0 and b > a:
                try:
                    v = json.loads(t[a:b + 1])
                except Exception:
                    return None
            else:
                return None
        if isinstance(v, str):
            t = v.strip()
            continue
        return v
    return None
verdict = _parse_verdict(raw) or {"status": "needs_review", "residual_hedges": None, "edits": ["unparseable verifier output"]}
status = verdict.get("status", "needs_review")
edits = verdict.get("edits", []) or []
hedges = verdict.get("residual_hedges")
print(f"STATUS: {status.upper()}   residual_hedges={hedges}   edits={len(edits)}")
for e in edits:
    print(" -", e)

## Step 6 — Persist the note

In [ ]:
session.sql(
    f"INSERT INTO {SCHEMA}.ATTRITION_REPORTS (subject, draft_md, report_md, status, edits_applied, residual_hedges) SELECT ?, ?, ?, ?, ?, ?",
    params=["Merchant attrition early-warning", draft_md, draft_md, status, len(edits), hedges]
).collect()
session.sql(f"SELECT run_id, subject, status, edits_applied, residual_hedges, created_at FROM {SCHEMA}.ATTRITION_REPORTS ORDER BY created_at DESC LIMIT 5").to_pandas()

## Recap & extensions

You built the verified attrition early-warning loop on free category-demand data. `setup.sql` already created an **`ATTRITION_AGENT`** — chat with it in Snowsight → AI & ML → Agents (*"which at-risk merchants should we save first, and is their category still growing?"*).

**Next:** point `ATTRITION_MERCHANT` / `ATTRITION_MERCHANT_MONTHLY` at your own merchant master + settlement/engagement data (same columns) to run this on your real book.

**Try breaking the verifier:** add a made-up projection (e.g., *"restaurants will likely lose 20% more merchants next quarter"*) to the draft and watch Step 5 flag it *Needs review*.